# Visualización de Filtros y Activaciones en Redes Convolucionales (CNNs)

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Asignatura Ministerial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

El objetivo de este laboratorio es desmitificar el comportamiento interno de las **Redes Neuronales Convolucionales (CNNs)**. Abriremos la "caja negra" del aprendizaje profundo para visualizar directamente cómo los filtros (*kernels*) aprenden a extraer características geométricas elementales (como bordes y contornos) y cómo se propagan e intensifican las respuestas de activación espacial a través de los mapas de características jerárquicos.

## Resultados de aprendizaje

Al final de este notebook van a poder:
1. Explicar el funcionamiento de una capa convolucional 2D y una capa de agrupación máxima (MaxPooling).
2. Extraer y visualizar los pesos y geometrías de los filtros aprendidos en las primeras etapas de una CNN.
3. Construir modelos de inspección intermedios en Keras para capturar las activaciones de capas ocultas.
4. Graficar y contrastar mapas de características en escala espacial antes y después del submuestreo espacial.
5. Interpretar la transición de características locales simples a representaciones espaciales abstractas.

## Terminología clave (Microglosario)

*   **Red Neuronal Convolucional (CNN):** Modelo neuronal diseñado específicamente para procesar datos estructurados en cuadrículas (como imágenes), explotando la correlación espacial local de los píxeles.
*   **Filtro (Kernel):** Pequeña matriz de parámetros ajustables que se desliza por la imagen de entrada para calcular productos internos locales y detectar patrones específicos (bordes, esquinas, texturas).
*   **Mapa de Características (Feature Map):** Representación bidimensional resultante de aplicar un filtro sobre una imagen, indicando las localizaciones espaciales donde se detectó el patrón de interés.
*   **MaxPooling (Agrupación Máxima):** Operación de submuestreo que reduce el ancho y alto espacial de los mapas de características al quedarse únicamente con el valor máximo de pequeñas ventanas de análisis.
*   **Función de Activación ReLU:** Función no lineal que convierte todos los valores de activación negativos en cero, permitiendo que la red modele relaciones de alta complejidad matemática.

## 1. Configuración del Entorno y Carga de Datos

Comencemos importando las librerías necesarias. Trabajaremos con el clásico dataset MNIST (dígitos manuscritos del 0 al 9), que nos servirá como baseline didáctico para analizar las estructuras visuales aprendidas por la CNN.

In [ ]:
# Importamos las dependencias esenciales de análisis y visualización
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

print("✓ Entorno de análisis e importaciones de TensorFlow listas.")

In [ ]:
# Cargamos el conjunto de datos MNIST directamente de la API de Keras
(imagenes_entrenamiento_brutas, etiquetas_entrenamiento), (imagenes_prueba_brutas, etiquetas_prueba) = tf.keras.datasets.mnist.load_data()

# Preprocesamiento didáctico paso a paso:
# 1. MNIST posee dimensiones de (muestras, 28, 28). Las CNNs de Keras exigen un canal de color explícito.
#    Añadimos un eje al final para indicar un único canal en escala de grises:
x_train_expandido = imagenes_entrenamiento_brutas[..., np.newaxis]
x_test_expandido = imagenes_prueba_brutas[..., np.newaxis]

# 2. Convertimos los píxeles a decimal y normalizamos dividiendo por 255.0 para dejarlos en el rango [0, 1]:
imagenes_entrenamiento = x_train_expandido / 255.0
imagenes_prueba = x_test_expandido / 255.0

print(f"✓ Datos preparados para la CNN:")
print(f"  • Dimensiones de entrenamiento: {imagenes_entrenamiento.shape}")
print(f"  • Dimensiones de prueba: {imagenes_prueba.shape}")

## 1.1. Inspección Visual de una Muestra de Entrada

Verifiquemos cómo se observa visualmente un dígito manuscrito después del preprocesamiento y escalado espacial.

In [ ]:
# Seleccionamos la primera muestra de entrenamiento para graficar
imagen_ejemplo = imagenes_entrenamiento[0, :, :, 0]
etiqueta_ejemplo = etiquetas_entrenamiento[0]

plt.figure(figsize=(3, 3))
plt.imshow(imagen_ejemplo, cmap='gray')
plt.title(f"Dígito Real: {etiqueta_ejemplo}", fontsize=11, fontweight="bold")
plt.axis('off')
plt.show()

## 2. Construcción y Entrenamiento de una CNN Simple

Diseñaremos una red convolucional secuencial pequeña para facilitar la inspección visual directa. Su arquitectura consta de:
1.  **Capa Convolucional (`Conv2D`):** Aplica filtros de $3 \times 3$ sobre los píxeles de entrada.
2.  **Capa de Agrupación (`MaxPooling2D`):** Submuestrea el espacio descartando activaciones secundarias.
3.  **Aplanado y Salida Densa:** Clasifica finalmente los mapas de características jerárquicos reducidos en probabilidades probabilísticas.

In [ ]:
# Definimos la arquitectura secuencial de la CNN
modelo_cnn = tf.keras.Sequential([
    # Capa convolucional: 16 filtros de tamaño 3x3 y activación ReLU
    tf.keras.layers.Conv2D(16, 3, activation='relu', input_shape=(28, 28, 1)),
    
    # Capa de submuestreo espacial MaxPooling
    tf.keras.layers.MaxPooling2D(),
    
    # Segunda capa convolucional: 32 filtros de tamaño 3x3 y activación ReLU
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    
    # Aplanado para conectar con la salida
    tf.keras.layers.Flatten(),
    
    # Capa de salida lineal para las 10 categorías (dígitos 0 al 9) con Softmax
    tf.keras.layers.Dense(10, activation='softmax')
])

In [ ]:
# Compilamos la CNN especificando pérdida multiclase y optimizador
modelo_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✦ Iniciando el entrenamiento de la CNN por 3 épocas. Por favor, aguarden...")
# Entrenamos reteniendo un 10% para validación interna amigable
modelo_cnn.fit(
    imagenes_entrenamiento, 
    etiquetas_entrenamiento, 
    epochs=3, 
    validation_split=0.1
)
print("\n✓ Entrenamiento completado.")

## 3. Visualización de Filtros Aprendidos (Kernels)

Los **filtros** son matrices pequeñas de coeficientes ajustables que la red convolucional aprende para extraer patrones de interés. Inspeccionaremos de forma directa la anatomía de los 16 filtros ajustados de la primera capa convolucional para observar qué tipo de bordes o esquinas detecta cada uno.

In [ ]:
# Extraemos los pesos y sesgos de la primera capa convolucional (índice 0)
primera_capa_pesos = modelo_cnn.layers[0].get_weights()
filtros_aprendidos = primera_capa_pesos[0]

print(f"✓ Forma de los filtros aprendidos: {filtros_aprendidos.shape}") # (alto, ancho, canales_entrada, cantidad_filtros)

# Graficamos la matriz de filtros en una cuadrícula de 4x4
figura_filtros, ejes_filtros = plt.subplots(4, 4, figsize=(7, 7))
figura_filtros.suptitle('Filtros Aprendidos en la Primera Capa Convolucional', fontsize=12, fontweight="bold")

for i, ax in enumerate(ejes_filtros.flat):
    # Extraemos los pesos del i-ésimo filtro (canal 0 ya que es escala de grises)
    matriz_filtro = filtros_aprendidos[:, :, 0, i]
    
    ax.imshow(matriz_filtro, cmap='gray')
    ax.set_title(f'Filtro {i+1}', fontsize=9)
    ax.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Consigna de Lectura e Interpretación

Observen la distribución de intensidades claras y oscuras en los filtros. ¿Pueden identificar algún filtro que parezca diseñado específicamente para detectar bordes estrictamente verticales u horizontales?

## 4. Captura y Visualización de Mapas de Características

A medida que una imagen ingresa a la CNN, los filtros se convolucionan con ella generando **mapas de características**. Para inspeccionar estas respuestas espaciales en vivo, construiremos un modelo intermedio en Keras que capture las activaciones de las primeras dos capas.

In [ ]:
# Extraemos las capas de interés de manera individual para evitar comprensiones complejas en una sola línea
capa_convolucional = modelo_cnn.layers[0]
capa_maxpooling = modelo_cnn.layers[1]

# Definimos las salidas que deseamos inspeccionar
salida_conv = capa_convolucional.output
salida_pool = capa_maxpooling.output

# Creamos un modelo intermedio que recibe la misma entrada y nos retorna ambas salidas
modelo_activaciones = tf.keras.Model(
    inputs=modelo_cnn.input, 
    outputs=[salida_conv, salida_pool]
)

# Extraemos una muestra de prueba individual
imagen_prueba_muestra = imagenes_prueba[0:1]
etiqueta_prueba_muestra = etiquetas_prueba[0]

# Ejecutamos la predicción intermedia
activaciones = modelo_activaciones.predict(imagen_prueba_muestra)
print(f"\n✓ Mapas de características extraídos para el dígito: {etiqueta_prueba_muestra}")

## 4.1. Inspección de Mapas Convolucionales (`Conv2D`)

Graficaremos las 16 respuestas de salida producidas por la primera capa de convolución. Las zonas con mayor brillo (intensidad de color) delatan dónde reaccionó el filtro con mayor fuerza.

In [ ]:
activacion_primera_capa = activaciones[0]
num_filtros = activacion_primera_capa.shape[-1]

figura_conv, ejes_conv = plt.subplots(4, 4, figsize=(7, 7))
figura_conv.suptitle(f'Activaciones de la Primera Capa Conv2D (Dígito Real: {etiquetas_prueba[0]})', fontsize=12, fontweight="bold")

for i, ax in enumerate(ejes_conv.flat):
    # Seleccionamos el mapa del i-ésimo filtro (lote 0)
    mapa_caracteristicas = activacion_primera_capa[0, :, :, i]
    
    # Viridis es un mapa de color ideal para resaltar la intensidad de activaciones
    ax.imshow(mapa_caracteristicas, cmap='viridis')
    ax.set_title(f'Filtro {i+1}', fontsize=9)
    ax.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Consigna de Lectura e Interpretación

Observen con cuidado las activaciones convolucionales. ¿Por qué algunos filtros resaltan con nitidez el contorno izquierdo del dígito mientras que otros destacan el contorno superior o inferior?

## 4.2. Inspección de Mapas Submuestreados (`MaxPooling2D`)

La capa de MaxPooling reduce a la mitad el tamaño del alto y ancho espacial de los mapas de características al retener únicamente el valor máximo de cada región local. Analizaremos cómo se preserva la información crítica bajo esta compresión espacial.

In [ ]:
activacion_segunda_capa = activaciones[1]

figura_pool, ejes_pool = plt.subplots(4, 4, figsize=(7, 7))
figura_pool.suptitle(f'Activaciones de la Capa MaxPooling2D (Dígito Real: {etiquetas_prueba[0]})', fontsize=12, fontweight="bold")

for i, ax in enumerate(ejes_pool.flat):
    mapa_submuestreado = activacion_segunda_capa[0, :, :, i]
    
    ax.imshow(mapa_submuestreado, cmap='viridis')
    ax.set_title(f'Mapa {i+1}', fontsize=9)
    ax.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Consigna de Lectura e Interpretación

Comparen los mapas convolucionales (Conv2D) con los mapas de agrupación (MaxPooling). ¿Qué cambios observan en la resolución y nitidez de las imágenes comprimidas? ¿Por qué esta pérdida de resolución espacial resulta beneficiosa para la clasificación?

## Conclusiones y Prácticas Sugeridas

Comprobaron visualmente cómo una CNN opera como un detector jerárquico de características espaciales locales. Los filtros se autoajustan para capturar rasgos primitivos (bordes y diagonales) y las capas de agrupación reducen la dimensionalidad física para proveer robustez e invarianza a traslaciones.

**Ejercicios de Experimentación Sugeridos:**
1. **Cambiar el Dígito de Entrada:** Seleccionen otro índice de prueba (por ejemplo `imagenes_prueba[5:6]`) y visualicen los cambios correspondientes en las activaciones convolucionales. ¿Qué filtros reaccionan con mayor potencia?
2. **Añadir Capas Convolucionales Profundas:** Extiendan la arquitectura con un bloque convolucional adicional (`Conv2D` y `MaxPooling`) y analicen cómo las activaciones en capas más profundas se tornan gradualmente abstractas y menos semejantes al dígito original.